In [ ]:
import os
import re
import torch
from transformers import AutoTokenizer, RobertaForSequenceClassification
from underthesea import word_tokenize

print("🚀 BẮT ĐẦU KHỞI ĐỘNG HỆ THỐNG DEMO LOCAL...")

# ==========================================
# 1. NẠP TỪ ĐIỂN TEENCODE (Đường dẫn tương đối)
# ==========================================
# File từ điển nằm cùng thư mục Notebooks với file code này
dict_path = 'acronyms_dictionary.txt' 
ACRONYM_DICT = {}

print("🔍 Đang nạp từ điển teencode...")
try:
    with open(dict_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip() 
            # Bỏ qua các dòng trống hoặc dòng comment bắt đầu bằng '#'
            if not line or line.startswith('#'):
                continue
            
            # Cắt dựa trên dấu '=' 
            parts = line.split('=')
            if len(parts) >= 2:
                key = parts[0].strip().lower()
                value = parts[1].strip().lower()
                ACRONYM_DICT[key] = value
                
    print(f"✅ Đã nạp thành công {len(ACRONYM_DICT)} từ lóng/viết tắt từ máy tính!")
except Exception as e:
    print(f"⚠️ Lỗi đọc file teencode: {e}")

# ==========================================
# 2. HÀM TIỀN XỬ LÝ DỮ LIỆU
# ==========================================
def preprocess_text(text):
    text = text.lower()
    
    # Chuyển teencode
    words = text.split()
    text = " ".join([ACRONYM_DICT.get(w, w) for w in words])
    
    # Làm sạch
    text = re.sub(r'[^\s\wáàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệóòỏõọôốồổỗộơớờởỡợíìỉĩịúùủũụưứừửữựýỳỷỹỵđ_]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tách từ (Word Segmentation)
    text = word_tokenize(text, format="text")
    
    return text

# ==========================================
# 3. NẠP MÔ HÌNH PHOBERT (Đường dẫn tương đối)
# ==========================================
print("🔍 Đang nạp mô hình PhoBERT từ thư mục models...")

# Lùi ra 1 thư mục (..) rồi vào thư mục models
sentiment_model_path = '../models/sentiment_model'
topic_model_path = '../models/topic_model'

# Kiểm tra xem đường dẫn có tồn tại thật không trước khi load
if not os.path.exists(sentiment_model_path) or not os.path.exists(topic_model_path):
    print("❌ Lỗi: Không tìm thấy thư mục model! Hãy kiểm tra lại cấu trúc file.")
else:
    # Ưu tiên dùng Card rời (CUDA) trên máy bạn nếu có, không thì tự động dùng CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    tokenizer = AutoTokenizer.from_pretrained(sentiment_model_path, local_files_only=True)
    
    model_sentiment = RobertaForSequenceClassification.from_pretrained(sentiment_model_path, local_files_only=True).to(device)
    model_sentiment.eval()
    
    model_topic = RobertaForSequenceClassification.from_pretrained(topic_model_path, local_files_only=True).to(device)
    model_topic.eval()
    print(f"✅ Lõi PhoBERT đã sẵn sàng trên thiết bị: {device}")

dict_sentiment = {0: '🔴 Tiêu cực', 1: '⚪ Trung tính', 2: '🟢 Tích cực'}
dict_topic = {0: '👨‍🏫 Giảng viên', 1: '📚 Chương trình', 2: '🏫 Cơ sở vật chất', 3: '❓ Khác'}

# ==========================================
# 4. CHẠY DEMO THỰC TẾ
# ==========================================
print("\n" + "="*50)
print("🎯 CHƯƠNG TRÌNH DEMO PHOBERT (LOCAL VERSION)")
print("Nhập 'q' hoặc 'exit' để thoát.")
print("="*50 + "\n")

while True:
    raw_text = input("✍️ Mời nhập câu nhận xét: ")
    
    if raw_text.lower() in ['q', 'exit', 'thoat', 'quit']:
        print("👋 Đã thoát chương trình Demo!")
        break
        
    if not raw_text.strip():
        continue

    # Tiền xử lý
    clean_text = preprocess_text(raw_text)
    
    # Dự đoán
    inputs = tokenizer(clean_text, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
    
    with torch.no_grad():
        out_s = model_sentiment(**inputs)
        out_t = model_topic(**inputs)
        
        pred_s_idx = torch.argmax(out_s.logits, dim=1).item()
        pred_t_idx = torch.argmax(out_t.logits, dim=1).item()
        
        label_s = int(model_sentiment.config.id2label[pred_s_idx])
        label_t = int(model_topic.config.id2label[pred_t_idx])
        
    print("-" * 50)
    print(f"Câu gốc        : {raw_text}")
    print(f"Câu sau xử lý  : {clean_text}")
    print(f"Chủ đề         : {dict_topic[label_t]}")
    print(f"Cảm xúc        : {dict_sentiment[label_s]}")
    print("-" * 50 + "\n")

f:\Workspace\PYTHON\Projects\FeedbackAnalysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 BẮT ĐẦU KHỞI ĐỘNG HỆ THỐNG DEMO LOCAL...
🔍 Đang nạp từ điển teencode...
✅ Đã nạp thành công 579 từ lóng/viết tắt từ máy tính!
🔍 Đang nạp mô hình PhoBERT từ thư mục models...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3768.08it/s]

✅ Lõi PhoBERT đã sẵn sàng trên thiết bị: cpu

🎯 CHƯƠNG TRÌNH DEMO PHOBERT (LOCAL VERSION)
Nhập 'q' hoặc 'exit' để thoát.



--------------------------------------------------
Câu gốc        : thầy đẹp trai
Câu sau xử lý  : thầy đẹp trai
Chủ đề         : 👨‍🏫 Giảng viên
Cảm xúc        : 🟢 Tích cực
--------------------------------------------------

